<a href="https://colab.research.google.com/github/sourendra-optiflux/Railway-Crew-Rostering-Optimization/blob/main/railway_crew_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install -q ortools


In [ ]:

from datetime import datetime, timedelta
from ortools.sat.python import cp_model

print("OR-Tools is ready!")


OR-Tools is ready!


In [ ]:

crew = {
    "A": {"initial_location": "Yajpur"},
    "B": {"initial_location": "Yajpur"},
    "C": {"initial_location": "Howrah"},
}


In [ ]:

duties = [
    {
        "id": "G1",
        "start": "2026-11-01 17:00",
        "start_location": "Howrah",
        "end_location": "Yajpur",
        "activity": "Howrah Mainline",
        "eligible": ["A", "B"],
    },
    {
        "id": "G2",
        "start": "2026-11-02 17:00",
        "start_location": "Yajpur",
        "end_location": "Howrah",
        "activity": "Yajpur Mainline",
        "eligible": ["A", "B"],
    },
    {
        "id": "G3",
        "start": "2026-11-02 17:00",
        "start_location": "Howrah",
        "end_location": "Yajpur",
        "activity": "Howrah Mainline",
        "eligible": ["A", "C"],
    },
    {
        "id": "G4",
        "start": "2026-11-03 17:00",
        "start_location": "Yajpur",
        "end_location": "Howrah",
        "activity": "Yajpur Mainline",
        "eligible": ["A", "C"],
    },
    {
        "id": "G5",
        "start": "2026-11-04 17:00",
        "start_location": "Howrah",
        "end_location": "Yajpur",
        "activity": "Howrah Mainline",
        "eligible": ["A", "B", "C"],
    },
]

for duty in duties:
    duty["start_dt"] = datetime.strptime(
        duty["start"], "%Y-%m-%d %H:%M"
    )
    duty["end_dt"] = duty["start_dt"] + timedelta(hours=12)

print(f"Crew members: {len(crew)}")
print(f"Uncovered duties: {len(duties)}")


Crew members: 3
Uncovered duties: 5


In [ ]:

model = cp_model.CpModel()

num_duties = len(duties)
people = list(crew.keys())

# x[p, i] = 1 if person p is assigned to duty i
x = {}

for p in people:
    for i, duty in enumerate(duties):
        x[p, i] = model.new_bool_var(f"assign_{p}_{duty['id']}")

        # Ineligible assignments are forbidden.
        if p not in duty["eligible"]:
            model.add(x[p, i] == 0)

# Constraint 1: Each duty gets at most one person.
for i in range(num_duties):
    model.add(sum(x[p, i] for p in people) <= 1)

# Constraint 2: At most one duty starting on each calendar day.
for p in people:
    dates = sorted({
        duties[i]["start_dt"].date()
        for i in range(num_duties)
    })

    for day in dates:
        duties_that_day = [
            i for i, d in enumerate(duties)
            if d["start_dt"].date() == day
        ]
        model.add(
            sum(x[p, i] for i in duties_that_day) <= 1
        )

# Constraint 3: Each person's assigned duties form a valid route.
MIN_REST_HOURS = 11

for p in people:
    arcs = []

    # Node 0 is the route's depot/start-end node.
    # Duty i is represented by node i + 1.
    assigned_count = sum(x[p, i] for i in range(num_duties))

    empty_route = model.new_bool_var(f"empty_route_{p}")
    model.add(assigned_count == 0).only_enforce_if(empty_route)
    model.add(assigned_count >= 1).only_enforce_if(empty_route.Not())

    # If no duties are assigned, the route is just node 0.
    arcs.append((0, 0, empty_route))

    for i, duty_i in enumerate(duties):
        node_i = i + 1

        # An unassigned duty uses its self-loop.
        # An assigned duty must be part of the route.
        arcs.append((node_i, node_i, x[p, i].Not()))

        # A duty can be the first duty only if it starts
        # at this person's initial location.
        if duty_i["start_location"] == crew[p]["initial_location"]:
            start_arc = model.new_bool_var(f"start_{p}_{i}")
            arcs.append((0, node_i, start_arc))

        # A duty can be the last duty.
        end_arc = model.new_bool_var(f"end_{p}_{i}")
        arcs.append((node_i, 0, end_arc))

    # Add a route arc i -> j only when j can follow i.
    for i, duty_i in enumerate(duties):
        for j, duty_j in enumerate(duties):
            if i == j:
                continue

            enough_rest = (
                duty_j["start_dt"]
                >= duty_i["end_dt"]
                + timedelta(hours=MIN_REST_HOURS)
            )

            same_location = (
                duty_i["end_location"]
                == duty_j["start_location"]
            )

            if enough_rest and same_location:
                transition = model.new_bool_var(
                    f"next_{p}_{i}_{j}"
                )
                arcs.append((i + 1, j + 1, transition))

    # Selected duties must form one valid route.
    model.add_circuit(arcs)

# Objective: maximize the number of covered duties.
model.maximize(
    sum(x[p, i] for p in people for i in range(num_duties))
)

print("Model created.")


Model created.


In [ ]:

solver = cp_model.CpSolver()

# A generous limit for this tiny example.
solver.parameters.max_time_in_seconds = 10

status = solver.solve(model)

print("Solver status:", solver.status_name(status))

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Duties covered:", int(round(solver.objective_value)))
    print("Total duties:", len(duties))
else:
    print("No feasible assignment found.")


Solver status: OPTIMAL
Duties covered: 4
Total duties: 5


In [ ]:

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    assigned_duties = set()

    for p in people:
        person_duties = [
            i for i in range(num_duties)
            if solver.value(x[p, i]) == 1
        ]

        person_duties.sort(
            key=lambda i: duties[i]["start_dt"]
        )

        print(f"\nCrew {p}")
        print("Initial location:", crew[p]["initial_location"])

        if not person_duties:
            print("No duties assigned.")
            continue

        for i in person_duties:
            d = duties[i]
            assigned_duties.add(i)

            print(
                f"{d['id']} | "
                f"{d['start_dt']} to {d['end_dt']} | "
                f"{d['start_location']} -> {d['end_location']} | "
                f"{d['activity']}"
            )

    print("\n--- Uncovered duties ---")

    uncovered = [
        duties[i]["id"]
        for i in range(num_duties)
        if i not in assigned_duties
    ]

    print(uncovered if uncovered else "None")



Crew A
Initial location: Yajpur
No duties assigned.

Crew B
Initial location: Yajpur
G2 | 2026-11-02 17:00:00 to 2026-11-03 05:00:00 | Yajpur -> Howrah | Yajpur Mainline
G5 | 2026-11-04 17:00:00 to 2026-11-05 05:00:00 | Howrah -> Yajpur | Howrah Mainline

Crew C
Initial location: Howrah
G3 | 2026-11-02 17:00:00 to 2026-11-03 05:00:00 | Howrah -> Yajpur | Howrah Mainline
G4 | 2026-11-03 17:00:00 to 2026-11-04 05:00:00 | Yajpur -> Howrah | Yajpur Mainline

--- Uncovered duties ---
['G1']


In [ ]:

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    errors = []

    for p in people:
        assigned = [
            duties[i]
            for i in range(num_duties)
            if solver.value(x[p, i]) == 1
        ]
        assigned.sort(key=lambda d: d["start_dt"])

        location = crew[p]["initial_location"]
        previous_end = None
        previous_day = None

        for d in assigned:
            # Eligibility
            if p not in d["eligible"]:
                errors.append(f"{p}: ineligible for {d['id']}")

            # Location continuity
            if d["start_location"] != location:
                errors.append(
                    f"{p}: location mismatch before {d['id']}"
                )

            # Rest and overlap
            if previous_end is not None:
                rest_hours = (
                    d["start_dt"] - previous_end
                ).total_seconds() / 3600

                if rest_hours < 11:
                    errors.append(
                        f"{p}: insufficient rest before {d['id']}"
                    )

            # One duty per calendar start day
            current_day = d["start_dt"].date()
            if current_day == previous_day:
                errors.append(
                    f"{p}: multiple duties starting on {current_day}"
                )

            # Update state after performing this duty
            location = d["end_location"]
            previous_end = d["end_dt"]
            previous_day = current_day

    # At most one person per duty
    for i, d in enumerate(duties):
        assigned_count = sum(
            solver.value(x[p, i]) for p in people
        )
        if assigned_count > 1:
            errors.append(f"{d['id']}: assigned more than once")

    if errors:
        print("VALIDATION FAILED")
        for error in errors:
            print("-", error)
    else:
        print("VALIDATION PASSED")
        print("All modeled constraints hold.")


VALIDATION PASSED
All modeled constraints hold.
